In [ ]:
# Librerías de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns


# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

df = pd.read_csv('./data_madrid/datos_limpios.csv')

print(f"Dimensiones: {df.shape}")
print(f"\nColumnas: {df.columns.tolist()}")
print(f"\nNaN: {df.isnull().sum().sum()}")
df.head()

In [ ]:
# Verificar que todo está en orden antes de entrenar
print("=== TIPOS DE DATOS ===")
print(df.dtypes)
print(f"\n=== ESTADÍSTICAS ===")
df.describe()

## Separar features y target

In [ ]:
# Target: lo que queremos predecir
y = np.log1p(df['PrecioActual'])

# Features: todo lo demás
X = df.drop(columns=['PrecioActual'])

print(f"Features (X): {X.shape} → {X.columns.tolist()}")
print(f"Target (y):   {y.shape}")

## Train/Test Split

Dividimos los datos en dos grupos:
- **Train (80%)**: el modelo aprende con estos datos
- **Test (20%)**: evaluamos el modelo con datos que NUNCA ha visto

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42)

print(f"Train: {X_train.shape[0]} filas ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test:  {X_test.shape[0]} filas ({X_test.shape[0]/len(X)*100:.0f}%)")

### Transformar variables categoricas

### 1- Variable: `zona` → Target Encoding

In [ ]:
# ---  Calcular la media de precio por zona en TRAIN ---
# Necesitamos juntar X_train con y_train temporalmente para calcular la media
train_con_precio = X_train.copy()
train_con_precio['PrecioActual'] = y_train

media_precio_por_zona = train_con_precio.groupby('zona')['PrecioActual'].mean()
# Resultado: {'arganzuela': 450000, 'barrio-de-salamanca': 1800000, ...}

# --- PASO 3: Aplicar el mapping a train y test ---
X_train['zona_encoded'] = X_train['zona'].map(media_precio_por_zona)
X_test['zona_encoded'] = X_test['zona'].map(media_precio_por_zona)

# --- PASO 4: Eliminar la columna original de texto ---
X_train = X_train.drop(columns=['zona'])
X_test = X_test.drop(columns=['zona'])

### Variable: `tipo_inmueble` → One-Hot Encoding

In [ ]:
# get_dummies crea columnas binarias automáticamente
# drop_first=True elimina una columna para evitar multicolinealidad perfecta
# (si tienes 6 tipos, con 5 columnas puedes inferir el 6º)

X_train = pd.get_dummies(X_train, columns=['tipo_inmueble'], drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=['tipo_inmueble'], drop_first=True)

# ⚠️ Alinear columnas: si en test no aparece algún tipo, puede faltar una columna
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)


### Variables: `ascensor` y `localizacion` → One-Hot Encoding de los 3 valores

In [ ]:
for col in ['ascensor', 'localizacion']:
    dummies = pd.get_dummies(X_train[col], prefix=col, drop_first=False)
    # Resultado: ascensor_N, ascensor_NO_APLICA, ascensor_S

    # Quitamos la columna original y añadimos las dummies
    X_train = pd.concat([X_train.drop(columns=[col]), dummies], axis=1)

# Lo mismo para test (y alinear columnas después)
for col in ['ascensor', 'localizacion']:
    dummies = pd.get_dummies(X_test[col], prefix=col, drop_first=False)
    X_test = pd.concat([X_test.drop(columns=[col]), dummies], axis=1)

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

### Variable: `plantaN` — Separar el NO_APLICA
En vez de poner 0 a los chalets (que confunde "planta baja" con "no tiene planta"), crear una columna separada que indica si el inmueble tiene planta o no.

In [ ]:
# Crear columna binaria: 1 si tiene planta, 0 si es chalet/casa
X_train['tiene_planta'] = (X_train['plantaN'] != 'NO_APLICA').astype(int)
X_test['tiene_planta']  = (X_test['plantaN']  != 'NO_APLICA').astype(int)

# Para los NO_APLICA, ponemos NaN (que luego imputamos con la mediana de pisos)
X_train['plantaN'] = pd.to_numeric(X_train['plantaN'].replace('NO_APLICA', float('nan')))
X_test['plantaN']  = pd.to_numeric(X_test['plantaN'].replace('NO_APLICA',  float('nan')))

# Imputar con la mediana de pisos (calculada sobre train)
mediana_planta_train = X_train.loc[X_train['tiene_planta'] == 1, 'plantaN'].median()
X_train['plantaN'] = X_train['plantaN'].fillna(mediana_planta_train)
X_test['plantaN']  = X_test['plantaN'].fillna(mediana_planta_train)

## Escalar los datos

In [ ]:
scaler = StandardScaler()

# fit_transform en train: calcula media/std Y transforma
X_train_scaled = scaler.fit_transform(X_train)

# transform en test: usa la media/std de train para transformar
X_test_scaled = scaler.transform(X_test)

# Convertir a DataFrame para mantener los nombres de las columnas
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_train.columns)

print("Antes de escalar (primeras 3 filas de train):")
print(X_train.head(3).to_string())
print("\nDespués de escalar:")
print(X_train_scaled.head(3).to_string())

## Entrenar Regresión Lineal

In [ ]:
# Crear el modelo
lr = LinearRegression()

# Entrenar: el modelo aprende los coeficientes con los datos de train
lr.fit(X_train_scaled, y_train)

# Predecir: el modelo usa lo aprendido para predecir precios del test
#y_pred_lr = lr.predict(X_test_scaled)

# --- CAMBIO 3: Al predecir, revertir el log ---
# El modelo predice log(precio), hay que convertirlo a euros reales
y_pred_lr_log    = lr.predict(X_test_scaled)
y_pred_lr_euros  = np.expm1(y_pred_lr_log)  # ← convertir de vuelta a euros

print("Modelo de Regresión Lineal entrenado.")
print(f"Intercepto (b): {lr.intercept_:,.0f}")
print(f"Coeficientes: {len(lr.coef_)}")

## Entrenar Ridge

In [ ]:
# Probar varios valores de alpha para ver cuál funciona mejor
alphas = [0.1, 1.0, 10.0, 100.0, 1000.0]
resultados_ridge = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    y_pred = ridge.predict(X_test_scaled)
    r2 = r2_score(y_test, y_pred)
    resultados_ridge.append({'alpha': alpha, 'r2': r2})
    print(f"Ridge (alpha={alpha:>7}): R² = {r2:.4f}")

# Elegir el mejor alpha
mejor = max(resultados_ridge, key=lambda x: x['r2'])
print(f"\nMejor alpha: {mejor['alpha']} con R² = {mejor['r2']:.4f}")

### Entrenar el modelo final con el mejor alpha

In [ ]:

# Entrenar Ridge con el mejor alpha
mejor_alpha = mejor['alpha']
ridge_final = Ridge(alpha=mejor_alpha)
ridge_final.fit(X_train_scaled, y_train)
#y_pred_ridge = ridge_final.predict(X_test_scaled)
y_pred_ridge_log   = ridge_final.predict(X_test_scaled)
y_pred_ridge_euros = np.expm1(y_pred_ridge_log)

print(f"Ridge final entrenado con alpha={mejor_alpha}")

### Evaluar y comparar

In [ ]:
def evaluar_modelo(nombre, y_real, y_predicho):
    """Calcula y muestra las métricas de un modelo."""
    r2 = r2_score(y_real, y_predicho)
    mae = mean_absolute_error(y_real, y_predicho)
    rmse = np.sqrt(mean_squared_error(y_real, y_predicho))

    print(f"=== {nombre} ===")
    print(f"  R²:   {r2:.4f}  ({r2*100:.1f}% de la variación explicada)")
    print(f"  MAE:  {mae:,.0f} €  (error medio)")
    print(f"  RMSE: {rmse:,.0f} €  (error medio penalizando grandes errores)")
    return {'modelo': nombre, 'R2': r2, 'MAE': mae, 'RMSE': rmse}

print("=" * 60)
res_lr = evaluar_modelo("Regresión Lineal", y_test, y_pred_lr)
print()
res_ridge = evaluar_modelo(f"Ridge (alpha={mejor_alpha})", y_test, y_pred_ridge)
print("=" * 60)

# Tabla comparativa
comparacion = pd.DataFrame([res_lr, res_ridge])
print("\n=== COMPARACIÓN ===")
display(comparacion)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (nombre, y_pred) in zip(axes, [("Regresión Lineal", y_pred_lr),
                                         (f"Ridge (α={mejor_alpha})", y_pred_ridge)]):
    ax.scatter(y_test, y_pred, alpha=0.3, s=10)
    # Línea diagonal: si el modelo fuera perfecto, todos los puntos estarían aquí
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([0, max_val], [0, max_val], 'r--', label='Predicción perfecta')
    ax.set_xlabel('Precio real (€)')
    ax.set_ylabel('Precio predicho (€)')
    ax.set_title(nombre)
    ax.legend()

plt.tight_layout()
plt.show()

## Paso 11: Interpretar los coeficientes

Esta es la ventaja principal de la regresión lineal sobre los modelos de árbol: puedes ver exactamente cuánto pesa cada feature.


In [ ]:

# Crear tabla de coeficientes
coeficientes = pd.DataFrame({
    'Feature': X.columns,
    'Coef_LR': lr.coef_,
    'Coef_Ridge': ridge_final.coef_
}).sort_values('Coef_LR', ascending=False)

print("=== COEFICIENTES (datos escalados) ===")
print("Un coeficiente positivo = esa feature SUBE el precio")
print("Un coeficiente negativo = esa feature BAJA el precio")
print("El valor absoluto indica la FUERZA del efecto\n")
display(coeficientes)

# Gráfico de barras
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (nombre, col) in zip(axes, [("Regresión Lineal", "Coef_LR"),
                                      ("Ridge", "Coef_Ridge")]):
    datos = coeficientes.sort_values(col)
    colores = ['red' if x < 0 else 'green' for x in datos[col]]
    ax.barh(datos['Feature'], datos[col], color=colores)
    ax.set_title(f'Coeficientes - {nombre}')
    ax.set_xlabel('Peso')
    ax.axvline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()